# Multimodal Hate Speech Detection on FBHM via VLM Ensemble

Clean pipeline matching the research paper. Three tracks (Qwen2.5-VL LoRA, CLIP cross-attention, Gemini 2.5 Flash) combined via a logistic-regression meta-classifier.

**Run order:**
1. Setup & imports
2. Track B — CLIP cross-attention (produces `ensemble_results.json`)
3. Track A — Qwen2.5-VL + LoRA (produces `tier3_zeroshot_results.json`)
4. Track C — Gemini 2.5 Flash (produces `tier3_gemini_results.json`)
5. Meta-ensemble (consumes all three)

All evaluation is on the 500-sample FBHM development set.

**Note:** Track A must run before the CLIP prediction-export cell, because that cell loads the Qwen zero-shot predictions to align ordering. If you hit a missing-file error, check the run order.

## 0. Setup & Imports

In [ ]:
!pip install -q augly nlpaug imbalanced-learn

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    print(root)
    break  # just top level

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from transformers import CLIPModel, CLIPProcessor, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import json, os, random, pickle, time
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from tqdm import tqdm
from collections import Counter

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DATA_PATH = '/kaggle/input/datasets/parthplc/facebook-hateful-meme-dataset/data'
IMG_DIR   = DATA_PATH + '/img'

print(f"Device: {DEVICE}, GPUs: {torch.cuda.device_count()}")
print(f"Files: {os.listdir(DATA_PATH)[:5]}")

## 1. Track A — Qwen2.5-VL 7B with LoRA

LoRA fine-tuning (r=16, alpha=32, q_proj+v_proj). Prompt forces a single-token HATEFUL / NOT_HATEFUL output. Produces `tier3_zeroshot_results.json` (used by both the CLIP export cell and the meta-ensemble).

In [ ]:
# ============================================================
# Cell LoRA-1 — Install PEFT + verify Qwen loads
# ============================================================
!pip install -q peft bitsandbytes

import torch
print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    mem = torch.cuda.get_device_properties(i).total_mem / 1e9
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {mem:.1f} GB")

In [ ]:
# ============================================================
# LoRA v2 FULL — 8500 samples, proper label masking
# Expected: ~5 hours for 3 epochs, F1 74-78%
# ============================================================
!pip install -q peft

import torch, json, os, time, random, numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score
from collections import Counter
from peft import LoraConfig, get_peft_model, TaskType

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.empty_cache()

DATA_PATH = '/kaggle/input/datasets/parthplc/facebook-hateful-meme-dataset/data'
IMG_DIR = DATA_PATH + '/img'

with open(f'{DATA_PATH}/train.jsonl') as f: train_data = [json.loads(l) for l in f]
with open(f'{DATA_PATH}/dev.jsonl') as f: dev_data = [json.loads(l) for l in f]

# FULL DATASET — no subset
random.shuffle(train_data)
print(f"Train: {len(train_data)} (FULL), Dev: {len(dev_data)}")
print(f"Labels: {Counter([d['label'] for d in train_data])}")

# Load Qwen
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
print(f"\nLoading {MODEL_ID}...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("✓ Model loaded")

# LoRA
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1, bias="none", task_type=TaskType.CAUSAL_LM))
model.print_trainable_parameters()

PROMPT = """Classify this meme as HATEFUL or NOT_HATEFUL.
A meme is HATEFUL if it attacks people based on race, ethnicity, religion, gender, sexual orientation, or disability.
The meme text is: "{text}"
Answer: """

def fix_image(img):
    arr = np.array(img)
    if len(arr.shape) == 2: arr = np.stack([arr]*3, axis=2)
    elif arr.shape[2] == 4: arr = arr[:, :, :3]
    return Image.fromarray(arr)

def make_train_inputs(item, img):
    label_text = "HATEFUL" if item['label'] == 1 else "NOT_HATEFUL"
    prompt_messages = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": PROMPT.format(text=item['text'])},
    ]}]
    prompt_only = processor.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True)
    prompt_inputs = processor(text=[prompt_only], images=[img],
                              return_tensors="pt", padding=True)
    prompt_len = prompt_inputs['input_ids'].shape[1]

    full_messages = [
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": PROMPT.format(text=item['text'])},
        ]},
        {"role": "assistant", "content": [
            {"type": "text", "text": label_text},
        ]},
    ]
    full_text = processor.apply_chat_template(full_messages, tokenize=False)
    full_inputs = processor(text=[full_text], images=[img],
                           return_tensors="pt", padding=True)
    labels = full_inputs['input_ids'].clone()
    labels[0, :prompt_len] = -100
    full_inputs['labels'] = labels
    return full_inputs

# Training
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=2e-5, weight_decay=0.01)

NUM_EPOCHS = 2  # 2 epochs on full data (epoch 3 overfit on subset)
GRAD_ACCUM = 4
best_f1 = 0.0

print(f"\nTraining {NUM_EPOCHS} epochs on {len(train_data)} samples")
print(f"Estimated time: ~{len(train_data)*NUM_EPOCHS*1.5/3600:.1f} hours")

for epoch in range(NUM_EPOCHS):
    model.train()
    random.shuffle(train_data)
    total_loss, n_ok, n_fail = 0, 0, 0
    optimizer.zero_grad()
    t0 = time.time()

    for i, item in enumerate(tqdm(train_data, desc=f"Ep {epoch+1}/{NUM_EPOCHS}")):
        try:
            img = fix_image(Image.open(os.path.join(IMG_DIR, os.path.basename(item['img']))))
            inputs = make_train_inputs(item, img)
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

            outputs = model(**inputs)
            loss = outputs.loss / GRAD_ACCUM
            loss.backward()
            total_loss += loss.item() * GRAD_ACCUM
            n_ok += 1

            if (i + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()

        except torch.cuda.OutOfMemoryError:
            n_fail += 1
            torch.cuda.empty_cache()
            optimizer.zero_grad()
            continue
        except Exception as e:
            n_fail += 1
            if n_fail <= 3: print(f"\n  Error {i}: {str(e)[:80]}")
            continue

        if (i + 1) % 500 == 0:
            avg = total_loss / max(n_ok, 1)
            eta = (time.time()-t0)/(i+1) * (len(train_data)-i-1) / 60
            print(f"\n  [{i+1}/{len(train_data)}] loss={avg:.4f} ok={n_ok} fail={n_fail} ETA={eta:.0f}min")

    optimizer.step()
    optimizer.zero_grad()
    torch.cuda.empty_cache()
    ep_loss = total_loss / max(n_ok, 1)
    ep_time = (time.time() - t0) / 60
    print(f"\n  Epoch {epoch+1}: loss={ep_loss:.4f} ok={n_ok} fail={n_fail} time={ep_time:.1f}min")

    # Dev eval
    print("  Evaluating...")
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for item in tqdm(dev_data, desc="Dev"):
            try:
                img = fix_image(Image.open(os.path.join(IMG_DIR, os.path.basename(item['img']))))
                messages = [{"role": "user", "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": PROMPT.format(text=item['text'])},
                ]}]
                text_input = processor.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True)
                inputs = processor(text=[text_input], images=[img],
                                 return_tensors="pt", padding=True)
                inputs = {k: v.to(model.device) for k, v in inputs.items()}
                out_ids = model.generate(**inputs, max_new_tokens=5, do_sample=False)
                trimmed = out_ids[0][inputs['input_ids'].shape[1]:]
                resp = processor.decode(trimmed, skip_special_tokens=True).strip().upper()
                pred = 0 if ("NOT_HATEFUL" in resp or "NOT HATEFUL" in resp) else (1 if "HATEFUL" in resp else 0)
            except:
                pred = 0
                torch.cuda.empty_cache()
            preds.append(pred)
            targets.append(item['label'])

    f1 = f1_score(targets, preds, average='macro')
    acc = accuracy_score(targets, preds)
    mark = " ✓" if f1 > best_f1 else ""
    print(f"  Ep {epoch+1}: acc={acc:.4f} f1={f1:.4f}{mark}")
    print(f"  Preds: {Counter(preds)}")

    if f1 > best_f1:
        best_f1 = f1
        model.save_pretrained('/kaggle/working/qwen_lora_full')
        with open('/kaggle/working/qwen_lora_full_preds.json', 'w') as f:
            json.dump({'epoch':epoch+1, 'f1':float(f1), 'acc':float(acc),
                       'predictions':preds, 'targets':targets}, f, indent=2)

print(f"\n{'='*60}")
print(f"DONE — FULL DATASET LORA")
print(f"Best F1: {best_f1*100:.2f}%")
print(f"Subset (2000) was: 71.47%")
print(f"Full (8500):       {best_f1*100:.2f}%")
print(f"Delta:             {(best_f1-0.7147)*100:+.2f}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# Cell T3-4 — Prompt design + 5-sample sanity check
# ============================================================
from qwen_vl_utils import process_vision_info

# Prompt design — the most important decision for Tier 3
# Keep it focused: describe the task, explain what "hateful" means
# in FBHM's definition, and force a single-token output.
ZERO_SHOT_PROMPT = """You are classifying internet memes for hate speech.

A meme is HATEFUL if it attacks, demeans, or dehumanizes a person or group based on:
- race, ethnicity, nationality, religion
- gender, sexual orientation, disability
- or if it uses threats, slurs, or calls for violence against any group

A meme is NOT HATEFUL if it is benign, humorous, political without attacking a group, or critical without dehumanizing.

The meme text (as written on the image) is: "{text}"

Look carefully at BOTH the image and the text together. Sometimes the image alone is benign and the text alone is benign, but their combination creates hateful meaning.

Respond with exactly one word: HATEFUL or NOT_HATEFUL"""


def build_message(image_path, text):
    return [{
        "role": "user",
        "content": [
            {"type": "image", "image": f"file://{image_path}"},
            {"type": "text",  "text": ZERO_SHOT_PROMPT.format(text=text)},
        ],
    }]


@torch.no_grad()
def classify_meme(image_path, text, max_new_tokens=8):
    """Returns (prediction: 0|1, raw_response: str)"""
    messages = build_message(image_path, text)
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    # Trim the prompt tokens
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out_ids)]
    response = processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

    # Parse response → label
    resp_upper = response.upper()
    if "NOT_HATEFUL" in resp_upper or "NOT HATEFUL" in resp_upper:
        pred = 0
    elif "HATEFUL" in resp_upper:
        pred = 1
    else:
        # Fallback: look for affirmative/negative keywords
        pred = 1 if any(k in resp_upper for k in ["YES", "HATE", "OFFENSIV"]) else 0
    return pred, response


# Sanity check on 5 diverse dev samples (mix of labels)
print("Running 5-sample sanity check...")
print("="*80)

# Pick 3 hateful + 2 non-hateful
hateful_idx    = [i for i, d in enumerate(dev_data) if d['label'] == 1][:3]
notHateful_idx = [i for i, d in enumerate(dev_data) if d['label'] == 0][:2]
sample_idx = hateful_idx + notHateful_idx

for i in sample_idx:
    item = dev_data[i]
    img_path = os.path.join(IMG_DIR, os.path.basename(item['img']))
    pred, response = classify_meme(img_path, item['text'])
    correct = "✓" if pred == item['label'] else "✗"
    print(f"\n{correct} dev[{i}]  true={item['label']}  pred={pred}")
    print(f"  text: {item['text'][:80]}")
    print(f"  response: {response!r}")

In [ ]:
# ============================================================
# Cell T3-5 — Zero-shot VLM evaluation on full dev set
# ~20-25 min on 2×T4
# ============================================================
import time

results_zeroshot = []
preds_zeroshot = []
targets = []

print(f"Running zero-shot VLM on {len(dev_data)} dev samples...")
print("(ETA ~20-25 min)")
t0 = time.time()

for i, item in enumerate(tqdm(dev_data, desc="Zero-shot")):
    img_path = os.path.join(IMG_DIR, os.path.basename(item['img']))
    try:
        pred, response = classify_meme(img_path, item['text'])
    except Exception as e:
        print(f"Error on dev[{i}]: {e}")
        pred, response = 0, "ERROR"

    results_zeroshot.append({
        'idx': i,
        'true': item['label'],
        'pred': pred,
        'response': response,
        'text': item['text'],
    })
    preds_zeroshot.append(pred)
    targets.append(item['label'])

elapsed = time.time() - t0
print(f"\n✓ Done in {elapsed/60:.1f} min ({elapsed/len(dev_data):.2f} sec/sample)")

# Metrics
acc_zs = accuracy_score(targets, preds_zeroshot)
f1_zs  = f1_score(targets, preds_zeroshot, average='macro')

print(f"\n{'='*60}")
print(f"Zero-shot VLM — Acc: {acc_zs:.4f}  Macro-F1: {f1_zs:.4f}")
print(f"{'='*60}")

# Prediction distribution
from collections import Counter
print(f"Predictions: {dict(Counter(preds_zeroshot))}")
print(f"Targets:     {dict(Counter(targets))}")

# Save results for later analysis
with open('/kaggle/working/tier3_zeroshot_results.json', 'w') as f:
    json.dump({
        'accuracy': float(acc_zs),
        'macro_f1': float(f1_zs),
        'predictions': preds_zeroshot,
        'targets': targets,
        'per_sample': results_zeroshot,
    }, f, indent=2)
print("\n✓ Results saved to /kaggle/working/tier3_zeroshot_results.json")

## 2. Track B — CLIP with Cross-Attention Fusion

Frozen CLIP encoders (768-d) + learned bidirectional cross-attention + MLP head, trained with focal loss (gamma=2). The final cell runs the trained model on the dev set and exports `clip_preds` to `ensemble_results.json`.

In [ ]:
# ============================================================
# Cell T1-1 — Tier 1 setup: CLIP + cross-attention + focal loss
# ============================================================
!pip install -q transformers accelerate

import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor, get_linear_schedule_with_warmup
from PIL import Image
import json, os, random, pickle
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from tqdm import tqdm
from collections import Counter

# Config
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLIP_MODEL     = "openai/clip-vit-large-patch14"
BATCH_SIZE     = 32       # T4 can handle bigger than 12 for frozen CLIP
NUM_EPOCHS     = 30       # MLP head trains fast, more epochs is cheap
LR             = 5e-5
WEIGHT_DECAY   = 0.01
MAX_LEN        = 77       # CLIP's tokenizer hard limit
SEED           = 42

# FBHM dataset path (same as before — adjust if your Kaggle mount differs)
DATA_PATH = '/kaggle/input/datasets/parthplc/facebook-hateful-meme-dataset/data'
IMG_DIR   = DATA_PATH + '/img'

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"Device: {DEVICE}")
print(f"CLIP model: {CLIP_MODEL}")
print(f"Files: {os.listdir(DATA_PATH)[:5]}")

In [ ]:
# ============================================================
# Cell T1-2 — CLIP-compatible dataset
# ============================================================
class CLIPMemeDataset(Dataset):
    """
    Returns raw PIL images and text strings. The CLIPProcessor
    handles normalization and tokenization in the extraction cell.
    """
    def __init__(self, jsonl_path, img_dir):
        self.data = [json.loads(l) for l in open(jsonl_path)]
        self.img_dir = img_dir

    def __len__(self): return len(self.data)

    @staticmethod
    def _fix(img):
        a = np.array(img)
        if a.ndim == 2: a = np.stack([a]*3, axis=2)
        elif a.shape[2] == 4: a = a[:, :, :3]
        return Image.fromarray(a)

    def __getitem__(self, idx):
        item = self.data[idx]
        img = self._fix(Image.open(os.path.join(self.img_dir, os.path.basename(item['img']))))
        return {
            'image': img,
            'text':  item['text'],
            'label': item.get('label', -1),
        }

train_ds = CLIPMemeDataset(f'{DATA_PATH}/train.jsonl', IMG_DIR)
dev_ds   = CLIPMemeDataset(f'{DATA_PATH}/dev.jsonl',   IMG_DIR)
print(f"Train: {len(train_ds)}  |  Dev: {len(dev_ds)}")

# Quick sanity check on one sample
s = train_ds[0]
print(f"Sample: image={s['image'].size}, text='{s['text'][:60]}', label={s['label']}")

In [ ]:
# ============================================================
# Cell T1-3 — Extract CLIP features (cache to disk)
# ~10 min on T4 for 9000 images
# ============================================================
CLIP_CACHE = '/kaggle/working/clip_features.pkl'

if os.path.exists(CLIP_CACHE):
    print("Loading cached CLIP features...")
    with open(CLIP_CACHE, 'rb') as f:
        tr_img, tr_txt, tr_y, dv_img, dv_txt, dv_y = pickle.load(f)
else:
    print("Extracting CLIP features (first time only)...")
    clip_model = CLIPModel.from_pretrained(CLIP_MODEL).to(DEVICE).eval()
    clip_proc  = CLIPProcessor.from_pretrained(CLIP_MODEL)

    @torch.no_grad()
    def extract(dataset, desc):
        img_feats, txt_feats, labels = [], [], []
        BS = 16
        for i in tqdm(range(0, len(dataset), BS), desc=desc):
            batch = [dataset[j] for j in range(i, min(i+BS, len(dataset)))]
            images = [b['image'] for b in batch]
            texts  = [b['text']  for b in batch]
            labs   = [b['label'] for b in batch]

            inputs = clip_proc(
                text=texts, images=images,
                return_tensors='pt', padding=True,
                truncation=True, max_length=MAX_LEN
            ).to(DEVICE)

            img_f = clip_model.get_image_features(pixel_values=inputs['pixel_values'])
            txt_f = clip_model.get_text_features(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask']
            )

            # Handle both tensor and ModelOutput return types
            if hasattr(img_f, 'pooler_output'):
                img_f = img_f.pooler_output
            elif hasattr(img_f, 'last_hidden_state'):
                img_f = img_f.last_hidden_state[:, 0, :]
            if hasattr(txt_f, 'pooler_output'):
                txt_f = txt_f.pooler_output
            elif hasattr(txt_f, 'last_hidden_state'):
                txt_f = txt_f.last_hidden_state[:, 0, :]

            img_f = F.normalize(img_f, dim=-1)
            txt_f = F.normalize(txt_f, dim=-1)

            img_feats.append(img_f.cpu().numpy())
            txt_feats.append(txt_f.cpu().numpy())
            labels.extend(labs)

        return np.vstack(img_feats), np.vstack(txt_feats), np.array(labels)

    tr_img, tr_txt, tr_y = extract(train_ds, "Train")
    dv_img, dv_txt, dv_y = extract(dev_ds,   "Dev")

    del clip_model, clip_proc
    torch.cuda.empty_cache()

    with open(CLIP_CACHE, 'wb') as f:
        pickle.dump((tr_img, tr_txt, tr_y, dv_img, dv_txt, dv_y), f)
    print(f"✓ Cached to {CLIP_CACHE}")

print(f"\nTrain: img {tr_img.shape}  txt {tr_txt.shape}  labels {tr_y.shape}")
print(f"Dev:   img {dv_img.shape}  txt {dv_txt.shape}  labels {dv_y.shape}")
print(f"Class balance train: {dict(Counter(tr_y))}")
print(f"Class balance dev:   {dict(Counter(dv_y))}")

In [ ]:
# ============================================================
# Cell T1-4 — Cross-attention fusion model + focal loss
# ============================================================
class CrossAttentionFusion(nn.Module):
    """
    Takes CLIP image and text features (both 768-dim for ViT-L/14)
    and fuses them with bidirectional cross-attention + MLP head.
    """
    def __init__(self, feat_dim=768, n_heads=8, dropout=0.3, num_classes=2):
        super().__init__()
        # Project CLIP features into a shared space (still 768)
        self.img_proj = nn.Linear(feat_dim, feat_dim)
        self.txt_proj = nn.Linear(feat_dim, feat_dim)

        # Bidirectional cross-attention
        self.img_to_txt = nn.MultiheadAttention(feat_dim, n_heads, dropout=dropout, batch_first=True)
        self.txt_to_img = nn.MultiheadAttention(feat_dim, n_heads, dropout=dropout, batch_first=True)

        self.norm1 = nn.LayerNorm(feat_dim)
        self.norm2 = nn.LayerNorm(feat_dim)

        # Classifier head on concatenated attended features
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim * 2, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, img_feat, txt_feat):
        # img_feat, txt_feat: (B, 768)
        img = self.img_proj(img_feat).unsqueeze(1)   # (B, 1, 768)
        txt = self.txt_proj(txt_feat).unsqueeze(1)   # (B, 1, 768)

        # Image attends to text
        img_attn, _ = self.img_to_txt(query=img, key=txt, value=txt)
        img_fused = self.norm1(img + img_attn).squeeze(1)   # (B, 768)

        # Text attends to image
        txt_attn, _ = self.txt_to_img(query=txt, key=img, value=img)
        txt_fused = self.norm2(txt + txt_attn).squeeze(1)   # (B, 768)

        # Concatenate and classify
        fused = torch.cat([img_fused, txt_fused], dim=-1)   # (B, 1536)
        return self.classifier(fused)


class FocalLoss(nn.Module):
    """
    Focal loss (Lin et al. 2017) — downweights easy examples,
    focuses gradient on hard minority cases. Drop-in for CrossEntropyLoss.
    """
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha  # optional class-weight tensor

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


# Sanity: instantiate and verify shapes
model = CrossAttentionFusion(feat_dim=tr_img.shape[1]).to(DEVICE)
test_img = torch.randn(4, tr_img.shape[1]).to(DEVICE)
test_txt = torch.randn(4, tr_txt.shape[1]).to(DEVICE)
with torch.no_grad():
    out = model(test_img, test_txt)
print(f"✓ Model output shape: {out.shape} (expected (4, 2))")
print(f"  Params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ============================================================
# Cell T1-5b — Better regularization + early stopping
# ============================================================
class RegularizedFusion(nn.Module):
    """Same architecture as CrossAttentionFusion but with feature dropout + stronger MLP dropout."""
    def __init__(self, feat_dim=768, n_heads=8, dropout=0.5, feat_dropout=0.2, num_classes=2):
        super().__init__()
        self.feat_drop = nn.Dropout(feat_dropout)   # NEW: drop feature dims
        self.img_proj = nn.Linear(feat_dim, feat_dim)
        self.txt_proj = nn.Linear(feat_dim, feat_dim)
        self.img_to_txt = nn.MultiheadAttention(feat_dim, n_heads, dropout=dropout, batch_first=True)
        self.txt_to_img = nn.MultiheadAttention(feat_dim, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(feat_dim)
        self.norm2 = nn.LayerNorm(feat_dim)
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim * 2, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, img_feat, txt_feat):
        img_feat = self.feat_drop(img_feat)
        txt_feat = self.feat_drop(txt_feat)
        img = self.img_proj(img_feat).unsqueeze(1)
        txt = self.txt_proj(txt_feat).unsqueeze(1)
        img_attn, _ = self.img_to_txt(query=img, key=txt, value=txt)
        img_fused = self.norm1(img + img_attn).squeeze(1)
        txt_attn, _ = self.txt_to_img(query=txt, key=img, value=img)
        txt_fused = self.norm2(txt + txt_attn).squeeze(1)
        fused = torch.cat([img_fused, txt_fused], dim=-1)
        return self.classifier(fused)


class FocalLossLS(nn.Module):
    """Focal loss with label smoothing."""
    def __init__(self, gamma=2.0, alpha=None, smoothing=0.1, num_classes=2):
        super().__init__()
        self.gamma, self.alpha, self.smoothing, self.nc = gamma, alpha, smoothing, num_classes
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha,
                             reduction='none', label_smoothing=self.smoothing)
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


# Retrain with tighter regularization
NUM_EPOCHS_V2 = 25
WD_V2         = 0.1         # bumped from 0.01
LR_V2         = 3e-5        # slightly lower
PATIENCE      = 6           # stop if no F1 improvement for 6 epochs

model = RegularizedFusion(feat_dim=tr_img.shape[1], dropout=0.5, feat_dropout=0.2).to(DEVICE)
criterion = FocalLossLS(gamma=2.0, alpha=alpha, smoothing=0.1)

optimizer = optim.AdamW(model.parameters(), lr=LR_V2, weight_decay=WD_V2)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * len(train_loader) * NUM_EPOCHS_V2),
    num_training_steps=len(train_loader) * NUM_EPOCHS_V2,
)

best_f1_v2, best_acc_v2, best_epoch_v2 = 0.0, 0.0, 0
save_path_v2 = '/kaggle/working/best_tier1_clip_v2.pt'
patience_counter = 0

for ep in range(NUM_EPOCHS_V2):
    model.train()
    total_loss = 0
    for img, txt, y in train_loader:
        img, txt, y = img.to(DEVICE), txt.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(img, txt), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for img, txt, y in dev_loader:
            logits = model(img.to(DEVICE), txt.to(DEVICE))
            preds.extend(logits.argmax(1).cpu().numpy())
            targets.extend(y.numpy())
    acc = accuracy_score(targets, preds)
    f1  = f1_score(targets, preds, average='macro')

    improved = f1 > best_f1_v2
    mark = " ✓" if improved else ""
    print(f"ep {ep+1:2d}: loss {total_loss/len(train_loader):.4f}  acc {acc:.4f}  f1 {f1:.4f}{mark}")

    if improved:
        best_f1_v2, best_acc_v2, best_epoch_v2 = f1, acc, ep+1
        torch.save(model.state_dict(), save_path_v2)
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {ep+1} (no improvement for {PATIENCE} epochs)")
            break

print(f"\n{'='*60}")
print(f"V2 best epoch {best_epoch_v2}: Acc {best_acc_v2:.4f}  F1 {best_f1_v2:.4f}")
print(f"V1 best was:          Acc 0.6980  F1 0.6980")
print(f"Delta: F1 {(best_f1_v2-0.6980)*100:+.2f} points")

In [ ]:
# ============================================================
# Cell 3 — Ensemble: fine-tuned CLIP + VLM zero-shot
# ============================================================
# Load VLM predictions from earlier session
with open('/kaggle/working/tier3_zeroshot_results.json') as f:
    zs_results = json.load(f)
vlm_preds   = zs_results['predictions']
vlm_targets = zs_results['targets']

# Load the fine-tuned CLIP checkpoint and run dev inference once
ft_model.load_state_dict(torch.load('/kaggle/working/best_clip_finetuned.pt'))
ft_model.eval()

clip_preds, clip_targets = [], []
with torch.no_grad():
    for batch in dev_ft_loader:
        pv   = batch['pixel_values'].to(DEVICE)
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        logits = ft_model(pv, ids, mask)
        clip_preds.extend(logits.argmax(1).cpu().numpy().tolist())
        clip_targets.extend(batch['label'].numpy().tolist())

# Sanity check ordering
assert clip_targets == vlm_targets, "Dev set ordering mismatch!"

# Ensemble strategies
ensemble_trust_clip = [cp if cp == vp else cp for cp, vp in zip(clip_preds, vlm_preds)]
ensemble_trust_vlm  = [cp if cp == vp else vp for cp, vp in zip(clip_preds, vlm_preds)]
ensemble_or         = [1 if (cp == 1 or vp == 1) else 0 for cp, vp in zip(clip_preds, vlm_preds)]
ensemble_and        = [1 if (cp == 1 and vp == 1) else 0 for cp, vp in zip(clip_preds, vlm_preds)]

def report(name, preds):
    acc = accuracy_score(clip_targets, preds)
    f1  = f1_score(clip_targets, preds, average='macro')
    pc  = Counter(preds)
    print(f"{name:<40}{acc*100:>8.2f}%{f1*100:>10.2f}%   preds: {dict(pc)}")

# Agreement analysis
agree = sum(1 for cp, vp in zip(clip_preds, vlm_preds) if cp == vp)
print(f"CLIP vs VLM agreement: {agree}/{len(clip_preds)} ({100*agree/len(clip_preds):.1f}%)\n")

print("="*80)
print(f"{'Method':<40}{'Acc':>8}{'Macro-F1':>12}")
print("-"*80)
report("CLIP fine-tuned (alone)",         clip_preds)
report("VLM zero-shot (alone)",           vlm_preds)
report("Ensemble: trust CLIP on tie",     ensemble_trust_clip)
report("Ensemble: trust VLM on tie",      ensemble_trust_vlm)
report("Ensemble OR (either says hate)",  ensemble_or)
report("Ensemble AND (both say hate)",    ensemble_and)
print("="*80)

# Save ensemble results
with open('/kaggle/working/ensemble_results.json', 'w') as f:
    json.dump({
        'clip_preds': clip_preds,
        'vlm_preds':  vlm_preds,
        'targets':    clip_targets,
        'agreement_rate': agree/len(clip_preds),
        'variants': {
            'clip_alone': {'f1': f1_score(clip_targets, clip_preds, average='macro'),
                           'acc': accuracy_score(clip_targets, clip_preds)},
            'vlm_alone':  {'f1': f1_score(clip_targets, vlm_preds, average='macro'),
                           'acc': accuracy_score(clip_targets, vlm_preds)},
            'trust_clip': {'f1': f1_score(clip_targets, ensemble_trust_clip, average='macro'),
                           'acc': accuracy_score(clip_targets, ensemble_trust_clip)},
            'trust_vlm':  {'f1': f1_score(clip_targets, ensemble_trust_vlm,  average='macro'),
                           'acc': accuracy_score(clip_targets, ensemble_trust_vlm)},
            'or':         {'f1': f1_score(clip_targets, ensemble_or,         average='macro'),
                           'acc': accuracy_score(clip_targets, ensemble_or)},
            'and':        {'f1': f1_score(clip_targets, ensemble_and,        average='macro'),
                           'acc': accuracy_score(clip_targets, ensemble_and)},
        }
    }, f, indent=2)
print("\n✓ Saved to /kaggle/working/ensemble_results.json")

## 3. Track C — Gemini 2.5 Flash (Zero-Shot)

Zero-shot chain-of-thought prompting via API, no fine-tuning. Produces `tier3_gemini_results.json`.

In [ ]:
# ============================================================
# Full 500-sample eval on Gemini 2.5 Flash
# ============================================================
import time
from collections import Counter
from sklearn.metrics import f1_score, accuracy_score
from tqdm import tqdm

results_gemini = []
preds_gemini   = []
targets_gemini = []
total_cost     = 0

print(f"Running {MODEL_ID} on {len(dev_data)} dev samples...")
t0 = time.time()

for i, item in enumerate(tqdm(dev_data, desc="Gemini Flash")):
    img_path = os.path.join(IMG_DIR, os.path.basename(item['img']))
    pred, raw, cost = classify_via_api(img_path, item['text'])
    total_cost += cost

    results_gemini.append({
        'idx': i,
        'true': item['label'],
        'pred': pred,
        'response': raw,
        'text': item['text'],
        'cost': cost,
    })
    preds_gemini.append(pred)
    targets_gemini.append(item['label'])

elapsed = time.time() - t0

acc_g = accuracy_score(targets_gemini, preds_gemini)
f1_g  = f1_score(targets_gemini, preds_gemini, average='macro')

print(f"\n{'='*70}")
print(f"Gemini 2.5 Flash — Acc: {acc_g*100:.2f}%  Macro-F1: {f1_g*100:.2f}%")
print(f"Time: {elapsed/60:.1f} min  |  Actual cost: ${total_cost:.3f}  |  Remaining: ${0.24-total_cost:.3f}")
print(f"{'='*70}")
print(f"Predictions: {dict(Counter(preds_gemini))}")
print(f"Targets:     {dict(Counter(targets_gemini))}")
print(f"\nComparison:")
print(f"  Qwen 7B zero-shot (local):  Acc 65.60%  F1 65.56%")
print(f"  Gemini 2.5 Flash (API):     Acc {acc_g*100:.2f}%  F1 {f1_g*100:.2f}%")
print(f"  Tier 1 CLIP best:           Acc 69.80%  F1 69.80%")

# Save
with open('/kaggle/working/tier3_gemini_results.json', 'w') as f:
    json.dump({
        'model':       MODEL_ID,
        'accuracy':    float(acc_g),
        'macro_f1':    float(f1_g),
        'total_cost':  float(total_cost),
        'predictions': preds_gemini,
        'targets':     targets_gemini,
        'per_sample':  results_gemini,
    }, f, indent=2)
print("\n✓ Saved to /kaggle/working/tier3_gemini_results.json")

## 4. Meta-Ensemble

Combines Track A + B + C predictions via majority vote, weighted vote, and a logistic-regression meta-classifier (5-fold CV). Reports the final result and saves `meta_ensemble_results.json`.

In [ ]:
# ============================================================
# META-ENSEMBLE: Combine existing Track A + B + C predictions
# Uses predictions you already have saved
# ~2 min to run
# ============================================================
import json, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from collections import Counter

# Load existing predictions
# Track A: Qwen zero-shot
with open('/kaggle/working/tier3_zeroshot_results.json') as f:
    track_a = json.load(f)

# Track C: Gemini Flash
with open('/kaggle/working/tier3_gemini_results.json') as f:
    track_c = json.load(f)

targets = track_a['targets']
pred_a = track_a['predictions']     # Qwen zero-shot
pred_c = track_c['predictions']     # Gemini Flash

# Track B: CLIP cross-attention
# Need to regenerate predictions — load from ensemble results
with open('/kaggle/working/ensemble_results.json') as f:
    ens = json.load(f)
pred_b = ens['clip_preds']          # CLIP fine-tuned predictions

# Verify alignment
assert len(targets) == len(pred_a) == len(pred_b) == len(pred_c) == 500
print(f"All 500 dev predictions loaded ✓")
print(f"  Track A (Qwen):   F1 {f1_score(targets, pred_a, average='macro')*100:.2f}%")
print(f"  Track B (CLIP):   F1 {f1_score(targets, pred_b, average='macro')*100:.2f}%")
print(f"  Track C (Gemini): F1 {f1_score(targets, pred_c, average='macro')*100:.2f}%")

# ============================================================
# Method 1: Simple voting (majority of 3)
# ============================================================
vote_preds = []
for a, b, c in zip(pred_a, pred_b, pred_c):
    vote = 1 if (a + b + c) >= 2 else 0
    vote_preds.append(vote)

vote_f1 = f1_score(targets, vote_preds, average='macro')
vote_acc = accuracy_score(targets, vote_preds)
print(f"\n--- Majority voting (3 tracks) ---")
print(f"  Acc: {vote_acc*100:.2f}%  F1: {vote_f1*100:.2f}%")

# ============================================================
# Method 2: Weighted voting (Gemini gets more weight)
# ============================================================
for w_a, w_b, w_c in [(1,1,2), (1,1,3), (0,1,2), (1,2,3)]:
    weighted = []
    for a, b, c in zip(pred_a, pred_b, pred_c):
        score = w_a*a + w_b*b + w_c*c
        threshold = (w_a + w_b + w_c) / 2
        weighted.append(1 if score > threshold else 0)
    f1_w = f1_score(targets, weighted, average='macro')
    print(f"  Weights ({w_a},{w_b},{w_c}): F1 {f1_w*100:.2f}%")

# ============================================================
# Method 3: Stacked features → Logistic Regression
# Uses leave-one-out style to avoid overfitting on 500 samples
# ============================================================
from sklearn.model_selection import cross_val_predict

X = np.column_stack([pred_a, pred_b, pred_c])
y = np.array(targets)

# Cross-validated predictions (5-fold)
lr = LogisticRegression(random_state=42, class_weight='balanced')
meta_preds = cross_val_predict(lr, X, y, cv=5)

meta_f1 = f1_score(y, meta_preds, average='macro')
meta_acc = accuracy_score(y, meta_preds)
print(f"\n--- Logistic Regression meta-classifier (5-fold CV) ---")
print(f"  Acc: {meta_acc*100:.2f}%  F1: {meta_f1*100:.2f}%")

# ============================================================
# Method 4: Oracle (best possible combination)
# Shows upper bound of what 3-track ensemble could achieve
# ============================================================
oracle = []
for a, b, c, t in zip(pred_a, pred_b, pred_c, targets):
    if a == t or b == t or c == t:
        oracle.append(t)
    else:
        oracle.append(a)  # all wrong, pick any

oracle_f1 = f1_score(targets, oracle, average='macro')
print(f"\n--- Oracle (upper bound if we always pick correct track) ---")
print(f"  F1: {oracle_f1*100:.2f}%")

# ============================================================
# Agreement analysis
# ============================================================
all_agree = sum(1 for a,b,c in zip(pred_a,pred_b,pred_c) if a==b==c)
ab_agree = sum(1 for a,b in zip(pred_a,pred_b) if a==b)
ac_agree = sum(1 for a,c in zip(pred_a,pred_c) if a==c)
bc_agree = sum(1 for b,c in zip(pred_b,pred_c) if b==c)

print(f"\n--- Agreement rates ---")
print(f"  All 3 agree: {all_agree}/500 ({100*all_agree/500:.1f}%)")
print(f"  Qwen+CLIP:   {ab_agree}/500 ({100*ab_agree/500:.1f}%)")
print(f"  Qwen+Gemini: {ac_agree}/500 ({100*ac_agree/500:.1f}%)")
print(f"  CLIP+Gemini: {bc_agree}/500 ({100*bc_agree/500:.1f}%)")

# ============================================================
# Final comparison
# ============================================================
print(f"\n{'='*60}")
print(f"FINAL COMPARISON")
print(f"{'='*60}")
print(f"  Track A alone (Qwen):      F1 {f1_score(targets,pred_a,average='macro')*100:.2f}%")
print(f"  Track B alone (CLIP):      F1 {f1_score(targets,pred_b,average='macro')*100:.2f}%")
print(f"  Track C alone (Gemini):    F1 {f1_score(targets,pred_c,average='macro')*100:.2f}%")
print(f"  Majority vote:             F1 {vote_f1*100:.2f}%")
print(f"  Meta-classifier (LR, CV):  F1 {meta_f1*100:.2f}%")
print(f"  Oracle upper bound:        F1 {oracle_f1*100:.2f}%")
print(f"{'='*60}")

# Save
with open('/kaggle/working/meta_ensemble_results.json', 'w') as f:
    json.dump({
        'track_a_f1': float(f1_score(targets,pred_a,average='macro')),
        'track_b_f1': float(f1_score(targets,pred_b,average='macro')),
        'track_c_f1': float(f1_score(targets,pred_c,average='macro')),
        'majority_vote_f1': float(vote_f1),
        'meta_lr_f1': float(meta_f1),
        'oracle_f1': float(oracle_f1),
    }, f, indent=2)
print("\n✓ Saved to meta_ensemble_results.json")